# [Module 01 -- Fundamentals] Installation and Your First Crew

> **MLCourse -- Agentic AI -- CrewAI Fundamentals**

> CrewAI is a lightweight, open-source framework for orchestrating multi-agent
> systems. Unlike LangChain or LangGraph, CrewAI is fully standalone -- no
> dependency on LangChain's ecosystem. It models collaboration as **Crews**
> (teams of agents), **Agents** (specialized workers), and **Tasks** (units of
> work). This module installs CrewAI, verifies the environment, and runs the
> smallest possible crew: one agent, one task, sequential process.

## What you'll learn

- How to install CrewAI and its optional tool packages.
- How to verify that every import works and Ollama is reachable.
- The three core classes: `Agent`, `Task`, `Crew`.
- How to build the minimal crew: 1 agent + 1 task + sequential kickoff.
- How to parse crew output from `crew.kickoff()`.

In [ ]:
# --- Standard library imports -------------------------------------------------
import os                            # Environment variable access after load_dotenv().
from pathlib import Path             # OOP paths for the track-root walk pattern.

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv       # Loads .env into os.environ.

# Walk up from the notebook's cwd until we hit the track root "03_agentic_ai".
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

# Jupyter plotting magic, guarded for non-IPython execution.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root resolved to:", TRACK)

## 1. Installing CrewAI

CrewAI ships as two packages:

- **`crewai`** -- core framework: `Agent`, `Task`, `Crew`, `Process`.
- **`crewai[tools]`** -- adds built-in tools like `ScrapeWebsiteTool`,
  `FileReadTool`, `SerperDevTool`, etc.

The install is a one-liner. If you already have it installed, pip will skip
re-downloading. We guard the call so the notebook stays runnable even if
you run cells out of order.

In [ ]:
# pip install crewai "crewai[tools]"   -- uncomment if not yet installed.

try:
    import crewai                          # Core: Agent, Task, Crew, Process.
    from crewai import Agent, Task, Crew, Process
    print("[OK] crewai imported successfully.  Version:", crewai.__version__)
except ImportError as e:
    print("[ERROR] crewai not installed. Run:  pip install crewai \"crewai[tools]\"")
    print("        Detail:", e)

## 2. Verifying crewai_tools

The `crewai[tools]` extra installs `crewai_tools`, which bundles ready-made
tool classes. We import a lightweight one (`FileReadTool`) just to confirm
the package is present. This does NOT read any file yet.

In [ ]:
try:
    from crewai_tools import FileReadTool
    print("[OK] crewai_tools imported.  FileReadTool is available.")
except ImportError as e:
    print("[WARN] crewai_tools not installed.")
    print("       Run:  pip install \"crewai[tools]\"")
    print("       Detail:", e)

## 3. Verifying Ollama connectivity

CrewAI can use any LLM provider. We use **ChatOllama** (local, free) as the
default throughout this track. Before building agents, confirm the server is
up and the model tag matches what `ollama list` shows.

In [ ]:
from langchain_ollama import ChatOllama    # CrewAI accepts LangChain chat models.

LLM_MODEL = "llama3.1:8b"                     # Must match `ollama list` output exactly.

llm = ChatOllama(model=LLM_MODEL)

try:
    response = llm.invoke("Reply with just the word OK.")
    print("[OK] Ollama responded:", response.content[:80])
except Exception as e:
    print("[WARN] Ollama unreachable. Start Ollama, then run: ollama pull", LLM_MODEL)
    print("       Detail:", e)

## 4. Your first Agent

An `Agent` in CrewAI is a role-playing entity with four key parameters:

| Parameter        | Purpose                                              |
|------------------|------------------------------------------------------|
| `role`           | Short title, e.g. "Research Analyst"                 |
| `goal`           | One-sentence objective the agent strives toward.     |
| `backstory`      | Paragraph of context that primes the LLM's behavior. |
| `llm`            | The language model powering this agent.               |

Additional useful params: `tools` (list of tool objects), `allow_delegation`
(bool), `max_iter` (int), `verbose` (bool). We cover those in Module 02.

In [ ]:
agent = Agent(
    role="Greeter",                                    # Short, descriptive title.
    goal="Say hello to the user in a friendly way.",   # What the agent tries to do.
    backstory=(                                        # Context that shapes the LLM persona.
        "You are a warm and welcoming assistant who "
        "loves greeting people with a smile."
    ),
    llm=llm,                                           # ChatOllama instance from section 3.
    allow_delegation=False,                            # This agent works alone -- no passing tasks.
    verbose=True,                                      # Print agent reasoning steps to stdout.
)

print("Agent created:", agent.role)

## 5. Your first Task

A `Task` wraps a unit of work for a specific agent. Key parameters:

| Parameter           | Purpose                                           |
|---------------------|---------------------------------------------------|
| `description`       | Natural-language instruction for the agent.       |
| `expected_output`   | What the output should look like (format hint).   |
| `agent`             | Which agent owns this task.                       |

Tasks can optionally take `context` (list of prior tasks whose outputs feed
in) and `callback` (function called on completion). Both are covered in
Module 03.

In [ ]:
task = Task(
    description="Greet the user by saying hello and introducing yourself.",
    expected_output="A short greeting string, 1-3 sentences.",
    agent=agent,       # This task is assigned to the Greeter agent above.
)

print("Task created for agent:", task.agent.role)

## 6. Your first Crew

A `Crew` bundles agents and tasks into a runnable unit. The `process`
parameter controls execution flow:

- `Process.sequential` -- tasks run one after another, each seeing prior outputs.
- `Process.hierarchical` -- a manager agent delegates sub-tasks automatically.

For the minimal crew we use **sequential** with one agent and one task.

In [ ]:
crew = Crew(
    agents=[agent],       # List of agents; here just one.
    tasks=[task],         # List of tasks; here just one.
    process=Process.sequential,    # Run tasks in listed order.
    verbose=True,                  # Print step-by-step execution logs.
)

print("Crew created with", len(crew.agents), "agent(s) and", len(crew.tasks), "task(s).")

## 7. Kickoff -- running the crew

`crew.kickoff()` is the single entry point that executes the entire crew. It
returns a `CrewOutput` object. The main result is in `.raw` (the raw string
from the final task) and `.pydantic` / `.json_dict` if you defined output
schemas on the task.

> **First-run latency:** the first call loads the model into memory (cold
> start). Subsequent calls are much faster.

In [ ]:
try:
    result = crew.kickoff()
    print("\n=== Crew Output (raw) ===")
    print(result.raw)
    print("\n=== Crew Output (token usage) ===")
    print("Prompt tokens   :", result.token_usage.get("prompt_tokens", "n/a"))
    print("Completion tokens:", result.token_usage.get("completion_tokens", "n/a"))
except Exception as e:
    print("[demo skipped] Ensure Ollama is running with model:", LLM_MODEL)
    print("       Detail:", e)

## 8. Parsing the output

`CrewOutput` provides several access patterns:

- `result.raw` -- the plain string from the last task.
- `result.pydantic` -- a Pydantic model if the task defined `output_pydantic`.
- `result.json_dict` -- a dict if the task defined `output_json`.
- `result.token_usage` -- dict with token counts per provider.

For simple tasks without structured output schemas, `.raw` is all you need.

In [ ]:
try:
    # Access the raw text output.
    print("Type of result :", type(result).__name__)
    print("Raw output     :", repr(result.raw))

    # token_usage is a dict; print its keys for reference.
    print("Token usage keys:", list(result.token_usage.keys()))
except NameError:
    print("[skipped] Run section 7 first to create 'result'.")

## 9. Minimal crew -- complete pattern in one block

Below is the entire Agent + Task + Crew + Kickoff pipeline condensed into a
single cell. This is the template you will reuse and expand in every
subsequent CrewAI module.

In [ ]:
from crewai import Agent, Task, Crew, Process

# 1. Define the LLM (local Ollama, free, no API key needed).
local_llm = ChatOllama(model="llama3.1:8b")

# 2. Create one agent with a role, goal, and backstory.
solo_agent = Agent(
    role="Math Tutor",
    goal="Explain one math concept clearly in under 50 words.",
    backstory="You are a patient tutor who makes math accessible.",
    llm=local_llm,
    allow_delegation=False,
    verbose=False,     # Set True to see reasoning traces.
)

# 3. Create one task with a description and expected output format.
solo_task = Task(
    description="Explain what a derivative is, in plain language.",
    expected_output="A 2-4 sentence explanation a beginner can understand.",
    agent=solo_agent,
)

# 4. Wrap in a Crew and kick off.
solo_crew = Crew(
    agents=[solo_agent],
    tasks=[solo_task],
    process=Process.sequential,
    verbose=False,
)

try:
    solo_result = solo_crew.kickoff()
    print("=== Math Tutor says ===")
    print(solo_result.raw)
except Exception as e:
    print("[demo skipped] Ensure Ollama is running with model: llama3.1:8b")
    print("       Detail:", e)

## 10. Key takeaways

| Concept          | CrewAI class | Key params                              |
|------------------|-------------|-----------------------------------------|
| Worker           | `Agent`     | role, goal, backstory, llm, tools       |
| Work unit        | `Task`      | description, expected_output, agent     |
| Orchestrator     | `Crew`      | agents, tasks, process, verbose         |
| Execute          | `kickoff()` | returns `CrewOutput` with `.raw`        |

- CrewAI is **standalone** -- no LangChain dependency required.
- The minimal crew is 1 agent + 1 task + `Process.sequential`.
- `kickoff()` blocks until the crew finishes; use `.raw` for the text result.
- Next module: deep-dive into Agent parameters (delegation, reasoning, config
  formats).